
# Fuerzas


In [1]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación: Fuerza de Roce Estático vs Cinético</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Ajusta la <strong>Fuerza Aplicada (F)</strong> para mover el bloque. Observa cómo cambia la fuerza de roce en el <strong>DCL</strong> (derecha) y revisa los valores exactos en el <strong>Panel de Datos</strong> inferior.
    </p>
    
    <!-- Controles Superiores -->
    <div style="display: flex; gap: 20px; margin-bottom: 15px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 15px; align-items: center;">
            <button id="play_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">▶ Reproducir</button>
            <button id="reset_btn_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">🔄 Reiniciar</button>
        </div>
        
        <div style="display: flex; gap: 20px; align-items: center;">
            <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc;">
                <label style="font-weight: 600; font-size: 0.85em; margin-bottom: 5px; color: #0d6efd;">Fuerza Aplicada F (N)</label>
                <input type="range" id="f_slider_UID" min="0" max="100" step="1" value="0" style="width: 200px;">
            </div>
        </div>
    </div>
    
    <!-- Canvas de la Simulación -->
    <div style="display: flex; justify-content: center; margin-bottom: 15px;">
        <canvas id="canvas_sim_UID" width="700" height="280" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <!-- Panel de Telemetría -->
    <div style="background: #ffffff; border: 1px solid #d6d8db; border-radius: 5px; padding: 15px; margin-bottom: 15px;">
        <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 10px; text-align: center;">
            <div style="background: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #e9ecef;">
                <div style="font-size: 0.75em; color: #6c757d; font-weight: bold; text-transform: uppercase;">Estado</div>
                <div id="lbl_estado_UID" style="font-size: 1em; font-weight: bold; color: #6c757d; margin-top: 5px;">Reposo</div>
            </div>
            <div style="background: #e7f1ff; padding: 10px; border-radius: 5px; border: 1px solid #b6d4fe;">
                <div style="font-size: 0.75em; color: #084298; font-weight: bold; text-transform: uppercase;">F Aplicada</div>
                <div id="lbl_F_UID" style="font-size: 1.1em; font-weight: bold; color: #0d6efd; margin-top: 5px; font-family: monospace;">0.0 N</div>
            </div>
            <div style="background: #f8d7da; padding: 10px; border-radius: 5px; border: 1px solid #f5c2c7;">
                <div style="font-size: 0.75em; color: #842029; font-weight: bold; text-transform: uppercase;">F Roce</div>
                <div id="lbl_fric_UID" style="font-size: 1.1em; font-weight: bold; color: #dc3545; margin-top: 5px; font-family: monospace;">0.0 N</div>
            </div>
            <div style="background: #e2e3e5; padding: 10px; border-radius: 5px; border: 1px solid #d3d6d8;">
                <div style="font-size: 0.75em; color: #41464b; font-weight: bold; text-transform: uppercase;">Aceleración</div>
                <div id="lbl_a_UID" style="font-size: 1.1em; font-weight: bold; color: #212529; margin-top: 5px; font-family: monospace;">0.00 m/s²</div>
            </div>
            <div style="background: #d1e7dd; padding: 10px; border-radius: 5px; border: 1px solid #badbcc;">
                <div style="font-size: 0.75em; color: #0f5132; font-weight: bold; text-transform: uppercase;">Velocidad</div>
                <div id="lbl_v_UID" style="font-size: 1.1em; font-weight: bold; color: #198754; margin-top: 5px; font-family: monospace;">0.00 m/s</div>
            </div>
        </div>
        <div style="text-align: center; font-size: 0.85em; color: #6c757d; margin-top: 10px;">
            m = 10 kg, g = 10 m/s², μ_s = 0.5 (Máx 50 N), μ_k = 0.4 (Const 40 N)
        </div>
    </div>
    
    <!-- Filtros del DCL -->
    <div style="font-size: 0.95em; padding: 10px 15px; background: #e2e3e5; border-radius: 5px; color: #383d41; display: flex; flex-wrap: wrap; gap: 15px; justify-content: center; user-select: none;">
        <strong style="margin-right: 10px;">Vectores DCL:</strong>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_F_UID" checked style="cursor: pointer;">
            <span><b style="color:#0d6efd;">▬ F</b> (Aplicada)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_fric_UID" checked style="cursor: pointer;">
            <span><b style="color:#dc3545;">▬ f</b> (Roce)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_N_UID" checked style="cursor: pointer;">
            <span><b style="color:#212529;">▬ N</b> (Normal)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_P_UID" checked style="cursor: pointer;">
            <span><b style="color:#6f42c1;">▬ P</b> (Peso)</span>
        </label>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_sim_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_play = document.getElementById('play_UID');
            const btn_reset = document.getElementById('reset_btn_UID');
            const f_slider = document.getElementById('f_slider_UID');
            
            // Labels HTML
            const lbl_F = document.getElementById('lbl_F_UID');
            const lbl_fric = document.getElementById('lbl_fric_UID');
            const lbl_a = document.getElementById('lbl_a_UID');
            const lbl_v = document.getElementById('lbl_v_UID');
            const lbl_estado = document.getElementById('lbl_estado_UID');
            
            // Checkboxes
            const chk_F = document.getElementById('chk_F_UID');
            const chk_fric = document.getElementById('chk_fric_UID');
            const chk_N = document.getElementById('chk_N_UID');
            const chk_P = document.getElementById('chk_P_UID');
            
            let isPlaying = true;
            let animationId;
            let lastTime = performance.now(); 
            
            // --- Físicas ---
            const m = 10.0;        
            const g = 10.0;        
            const P = m * g;       // 100 N
            const N = P;           // 100 N
            
            const mu_s = 0.5;      
            const mu_k = 0.4;      
            
            const f_s_max = mu_s * N; // 50 N
            const f_k = mu_k * N;     // 30 N
            
            let F_app = 0.0;       
            let f_fric = 0.0;      
            let a = 0.0;           
            let v = 0.0;           
            let x = 50.0;          // Posición X
            
            let estado_txt = "Reposo";
            let color_estado = "#6c757d";

            function drawArrow(context, fromx, fromy, tox, toy, color, lineWidth=3) {
                const dx = tox - fromx;
                const dy = toy - fromy;
                const len = Math.hypot(dx, dy); // Calcula la longitud del vector
                
                // Si la longitud es casi cero, no dibujamos nada para no ensuciar el DCL
                if (len < 0.5) return; 
                
                // AQUÍ ESTÁ LA CORRECCIÓN: La punta no superará la mitad del tamaño de la flecha, max 10px.
                const headlen = Math.min(10, len * 0.5); 
                const angle = Math.atan2(dy, dx);
                
                context.beginPath();
                context.strokeStyle = color;
                context.lineWidth = lineWidth;
                context.moveTo(fromx, fromy);
                context.lineTo(tox, toy);
                context.stroke();
                
                context.beginPath();
                context.fillStyle = color;
                context.moveTo(tox, toy);
                context.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                context.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                context.fill();
            }

            function updatePhysics(dt) {
                F_app = parseFloat(f_slider.value);
                
                if (Math.abs(v) < 0.01) { 
                    v = 0; 
                    if (F_app <= f_s_max) {
                        f_fric = F_app; 
                        a = 0;
                        estado_txt = "Estático";
                        color_estado = "#6c757d";
                    } else {
                        f_fric = f_k;   
                        a = (F_app - f_fric) / m;
                        estado_txt = "Acelerando";
                        color_estado = "#dc3545";
                    }
                } else {
                    f_fric = f_k; 
                    a = (F_app - f_fric) / m;
                    
                    if (Math.abs(a) < 0.01) {
                        a = 0;
                        estado_txt = "Vel Constante";
                        color_estado = "#198754";
                    } else if (a > 0) {
                        estado_txt = "Acelerando";
                        color_estado = "#dc3545";
                    } else {
                        estado_txt = "Desacelerando";
                        color_estado = "#ffc107";
                    }
                }
                
                v += a * dt;
                
                if (v < 0) {
                    v = 0;
                    a = 0;
                }
                
                x += v * dt * 25; 
                if (x > canvas.width / 2 - 30) {
                    x = 50; 
                }
                
                // Actualizar Panel de Datos (HTML)
                lbl_F.innerText = F_app.toFixed(1) + " N";
                lbl_fric.innerText = f_fric.toFixed(1) + " N";
                lbl_a.innerText = a.toFixed(2) + " m/s²";
                lbl_v.innerText = v.toFixed(2) + " m/s";
                lbl_estado.innerText = estado_txt;
                lbl_estado.style.color = color_estado;
            }

            function drawSim() {
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                
                // --- MITAD IZQUIERDA: BLOQUE Y SUPERFICIE ---
                const floorY = canvas.height - 40;
                
                // Superficie
                ctx.beginPath();
                ctx.strokeStyle = "#888";
                ctx.lineWidth = 4;
                ctx.moveTo(10, floorY);
                ctx.lineTo(canvas.width / 2, floorY);
                ctx.stroke();
                
                ctx.beginPath();
                ctx.strokeStyle = "#bbb";
                ctx.lineWidth = 1;
                for (let i = 10; i < canvas.width / 2; i += 10) {
                    ctx.moveTo(i, floorY);
                    ctx.lineTo(i - 8, floorY + 10);
                }
                ctx.stroke();

                // Bloque
                const blockW = 60;
                const blockH = 40;
                ctx.fillStyle = "#ff8c00"; 
                ctx.strokeStyle = "#c06b00";
                ctx.lineWidth = 2;
                ctx.fillRect(x, floorY - blockH, blockW, blockH);
                ctx.strokeRect(x, floorY - blockH, blockW, blockH);
                ctx.fillStyle = "white";
                ctx.font = "italic 20px serif";
                ctx.fillText("m", x + blockW/2 - 10, floorY - 12);
                
                // Divisor Central
                ctx.beginPath();
                ctx.strokeStyle = "#e0e0e0";
                ctx.setLineDash([5, 5]);
                ctx.moveTo(canvas.width / 2 + 10, 20);
                ctx.lineTo(canvas.width / 2 + 10, canvas.height - 20);
                ctx.stroke();
                ctx.setLineDash([]);
                
                // --- MITAD DERECHA: DCL LIMPIO ---
                const dcl_x = canvas.width * 0.76;
                const dcl_y = canvas.height / 2;
                const sf = 1.1; // Factor de escala para N
                
                // Ejes DCL
                ctx.beginPath();
                ctx.strokeStyle = "#ddd";
                ctx.setLineDash([4, 4]);
                ctx.moveTo(dcl_x - 130, dcl_y); ctx.lineTo(dcl_x + 130, dcl_y);
                ctx.moveTo(dcl_x, dcl_y - 120); ctx.lineTo(dcl_x, dcl_y + 120);
                ctx.stroke();
                ctx.setLineDash([]);
                
                // Vectores
                if (chk_N.checked) drawArrow(ctx, dcl_x, dcl_y, dcl_x, dcl_y - N * sf, "#212529", 3); 
                if (chk_P.checked) drawArrow(ctx, dcl_x, dcl_y, dcl_x, dcl_y + P * sf, "#6f42c1", 3); 
                if (chk_fric.checked && f_fric > 0) drawArrow(ctx, dcl_x, dcl_y, dcl_x - f_fric * sf, dcl_y, "#dc3545", 3); 
                if (chk_F.checked && F_app > 0) drawArrow(ctx, dcl_x, dcl_y, dcl_x + F_app * sf, dcl_y, "#0d6efd", 3); 
                
                // Etiquetas de Vectores (Cortas)
                ctx.font = "bold 14px sans-serif";
                if (chk_N.checked) { ctx.fillStyle = "#212529"; ctx.fillText("N", dcl_x + 8, dcl_y - N * sf + 15); }
                if (chk_P.checked) { ctx.fillStyle = "#6f42c1"; ctx.fillText("P", dcl_x + 8, dcl_y + P * sf - 5); }
                if (chk_fric.checked && f_fric > 0) { ctx.fillStyle = "#dc3545"; ctx.fillText("f", dcl_x - f_fric * sf, dcl_y + 20); }
                if (chk_F.checked && F_app > 0) { ctx.fillStyle = "#0d6efd"; ctx.fillText("F", dcl_x + F_app * sf - 10, dcl_y - 10); }
            }

            function animate(timestamp) {
                let dt = (timestamp - lastTime) / 1000;
                if (dt > 0.1) dt = 0.1; 
                lastTime = timestamp;
                
                if (isPlaying) {
                    updatePhysics(dt);
                }
                
                drawSim();
                animationId = requestAnimationFrame(animate);
            }

            f_slider.addEventListener('input', () => {
                if (!isPlaying) {
                    updatePhysics(0);
                    drawSim();
                }
            });

            btn_play.addEventListener('click', () => {
                isPlaying = !isPlaying;
                if (isPlaying) {
                    btn_play.innerText = "⏸ Pausar";
                    btn_play.style.backgroundColor = "#ffc107";
                    btn_play.style.color = "#000";
                    lastTime = performance.now();
                } else {
                    btn_play.innerText = "▶ Reproducir";
                    btn_play.style.backgroundColor = "#198754";
                    btn_play.style.color = "#fff";
                }
            });

            btn_reset.addEventListener('click', () => {
                v = 0;
                x = 50;
                f_slider.value = 0;
                if (!isPlaying) {
                    updatePhysics(0);
                    drawSim();
                }
            });
            
            [chk_F, chk_fric, chk_N, chk_P].forEach(chk => {
                chk.addEventListener('change', () => {
                    if (!isPlaying) drawSim();
                });
            });
            
            btn_play.innerText = "⏸ Pausar";
            btn_play.style.backgroundColor = "#ffc107";
            btn_play.style.color = "#000";
            requestAnimationFrame(animate);
        }
        
        initSimulation();
    })();
</script>
"""

html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))

In [2]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación: Plano Inclinado y Fricción</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Ajusta el <strong>Ángulo de Inclinación (θ)</strong>. Si superas el ángulo crítico, el bloque resbalará. Al llegar al final de la rampa reaparecerá arriba manteniendo su velocidad (bucle continuo).
    </p>
    
    <!-- Controles Superiores -->
    <div style="display: flex; gap: 20px; margin-bottom: 15px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 15px; align-items: center;">
            <button id="play_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">▶ Reproducir</button>
            <button id="reset_btn_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold;">🔄 Reiniciar</button>
        </div>
        
        <div style="display: flex; gap: 20px; align-items: center;">
            <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc;">
                <label style="font-weight: 600; font-size: 0.85em; margin-bottom: 5px; color: #0d6efd;">Ángulo de Inclinación (θ)</label>
                <div style="display: flex; align-items: center; gap: 10px;">
                    <input type="range" id="ang_slider_UID" min="0" max="45" step="0.1" value="0" style="width: 180px;">
                    <span id="ang_val_UID" style="font-weight: bold; font-family: monospace; font-size: 0.9em; min-width: 45px;">0.0°</span>
                </div>
            </div>
        </div>
    </div>
    
    <!-- Canvas de la Simulación -->
    <div style="display: flex; justify-content: center; margin-bottom: 15px;">
        <canvas id="canvas_sim_UID" width="700" height="300" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <!-- Panel de Telemetría -->
    <div style="background: #ffffff; border: 1px solid #d6d8db; border-radius: 5px; padding: 15px; margin-bottom: 15px;">
        <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap: 10px; text-align: center;">
            <div style="background: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #e9ecef;">
                <div style="font-size: 0.7em; color: #6c757d; font-weight: bold; text-transform: uppercase;">Estado</div>
                <div id="lbl_estado_UID" style="font-size: 1em; font-weight: bold; color: #6c757d; margin-top: 5px;">Reposo</div>
            </div>
            <div style="background: #e7f1ff; padding: 10px; border-radius: 5px; border: 1px solid #b6d4fe;">
                <div style="font-size: 0.7em; color: #084298; font-weight: bold; text-transform: uppercase;">Normal (N)</div>
                <div id="lbl_N_UID" style="font-size: 1.1em; font-weight: bold; color: #212529; margin-top: 5px; font-family: monospace;">100.0 N</div>
            </div>
            <div style="background: #fff3cd; padding: 10px; border-radius: 5px; border: 1px solid #ffecb5;">
                <div style="font-size: 0.7em; color: #997404; font-weight: bold; text-transform: uppercase;">Peso en X (Px)</div>
                <div id="lbl_Px_UID" style="font-size: 1.1em; font-weight: bold; color: #fd7e14; margin-top: 5px; font-family: monospace;">0.0 N</div>
            </div>
            <div style="background: #f8d7da; padding: 10px; border-radius: 5px; border: 1px solid #f5c2c7;">
                <div style="font-size: 0.7em; color: #842029; font-weight: bold; text-transform: uppercase;">Fuerza Roce</div>
                <div id="lbl_fric_UID" style="font-size: 1.1em; font-weight: bold; color: #dc3545; margin-top: 5px; font-family: monospace;">0.0 N</div>
            </div>
            <div style="background: #e2e3e5; padding: 10px; border-radius: 5px; border: 1px solid #d3d6d8;">
                <div style="font-size: 0.7em; color: #41464b; font-weight: bold; text-transform: uppercase;">Aceleración</div>
                <div id="lbl_a_UID" style="font-size: 1.1em; font-weight: bold; color: #212529; margin-top: 5px; font-family: monospace;">0.00 m/s²</div>
            </div>
            <div style="background: #d1e7dd; padding: 10px; border-radius: 5px; border: 1px solid #badbcc;">
                <div style="font-size: 0.7em; color: #0f5132; font-weight: bold; text-transform: uppercase;">Velocidad</div>
                <div id="lbl_v_UID" style="font-size: 1.1em; font-weight: bold; color: #198754; margin-top: 5px; font-family: monospace;">0.00 m/s</div>
            </div>
        </div>
        <div style="text-align: center; font-size: 0.85em; color: #6c757d; margin-top: 10px;">
            m = 10 kg, g = 10 m/s², μ_s = 0.5 (θ_crit = 26.6°), μ_k = 0.4 
        </div>
    </div>
    
    <!-- Filtros del DCL -->
    <div style="font-size: 0.95em; padding: 10px 15px; background: #e2e3e5; border-radius: 5px; color: #383d41; display: flex; flex-wrap: wrap; gap: 15px; justify-content: center; user-select: none;">
        <strong style="margin-right: 10px;">Vectores DCL:</strong>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_N_UID" checked style="cursor: pointer;">
            <span><b style="color:#212529;">▬ N</b> (Normal)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_fric_UID" checked style="cursor: pointer;">
            <span><b style="color:#dc3545;">▬ f</b> (Roce)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_P_UID" checked style="cursor: pointer;">
            <span><b style="color:#6f42c1;">▬ P</b> (Peso Total)</span>
        </label>
        <label style="cursor: pointer; display: flex; align-items: center; gap: 5px;">
            <input type="checkbox" id="chk_PxPy_UID" style="cursor: pointer;">
            <span><b style="color:#fd7e14;">▬ Px, Py</b> (Componentes)</span>
        </label>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_sim_UID');
            if (!canvas) { setTimeout(initSimulation, 50); return; }
            
            const ctx = canvas.getContext('2d');
            const btn_play = document.getElementById('play_UID');
            const btn_reset = document.getElementById('reset_btn_UID');
            
            // Sliders y Labels
            const ang_slider = document.getElementById('ang_slider_UID');
            const ang_val = document.getElementById('ang_val_UID');
            const lbl_N = document.getElementById('lbl_N_UID');
            const lbl_Px = document.getElementById('lbl_Px_UID');
            const lbl_fric = document.getElementById('lbl_fric_UID');
            const lbl_a = document.getElementById('lbl_a_UID');
            const lbl_v = document.getElementById('lbl_v_UID');
            const lbl_estado = document.getElementById('lbl_estado_UID');
            
            // Checkboxes
            const chk_N = document.getElementById('chk_N_UID');
            const chk_fric = document.getElementById('chk_fric_UID');
            const chk_P = document.getElementById('chk_P_UID');
            const chk_PxPy = document.getElementById('chk_PxPy_UID');
            
            let isPlaying = true;
            let animationId;
            let lastTime = performance.now(); 
            
            // --- Físicas Base ---
            const m = 10.0;        
            const g = 10.0;        
            const P = m * g;       // 100 N
            
            const mu_s = 0.5;      
            const mu_k = 0.4;      
            
            let theta_deg = 0.0;
            let theta_rad = 0.0;
            let N_val = P;
            let P_x = 0;
            let f_fric = 0.0;      
            let a = 0.0;           
            let v = 0.0;           
            let s = 40.0; // Posición a lo largo de la pendiente
            
            let estado_txt = "Reposo";
            let color_estado = "#6c757d";

            function drawArrow(context, fromx, fromy, tox, toy, color, lineWidth=3, dashed=false) {
                const dx = tox - fromx;
                const dy = toy - fromy;
                const len = Math.hypot(dx, dy); 
                
                if (len < 0.5) return; 
                
                // PUNTA DE FLECHA ADAPTATIVA
                const headlen = Math.min(10, len * 0.5); 
                const angle = Math.atan2(dy, dx);
                
                context.beginPath();
                context.strokeStyle = color;
                context.lineWidth = lineWidth;
                if(dashed) context.setLineDash([5, 5]);
                context.moveTo(fromx, fromy);
                context.lineTo(tox, toy);
                context.stroke();
                context.setLineDash([]);
                
                context.beginPath();
                context.fillStyle = color;
                context.moveTo(tox, toy);
                context.lineTo(tox - headlen * Math.cos(angle - Math.PI / 6), toy - headlen * Math.sin(angle - Math.PI / 6));
                context.lineTo(tox - headlen * Math.cos(angle + Math.PI / 6), toy - headlen * Math.sin(angle + Math.PI / 6));
                context.fill();
            }

            function updatePhysics(dt) {
                theta_deg = parseFloat(ang_slider.value);
                theta_rad = theta_deg * Math.PI / 180.0;
                ang_val.innerText = theta_deg.toFixed(1) + "°";
                
                // Descomposición del peso
                N_val = P * Math.cos(theta_rad);
                P_x = P * Math.sin(theta_rad);
                
                let f_s_max = mu_s * N_val;
                let f_k = mu_k * N_val;
                
                if (Math.abs(v) < 0.01) { 
                    v = 0; 
                    if (P_x <= f_s_max) {
                        f_fric = P_x;  // Roce estático iguala a Px
                        a = 0;
                        estado_txt = "Estático";
                        color_estado = "#6c757d";
                    } else {
                        f_fric = f_k;  // Rompe estático
                        a = (P_x - f_fric) / m;
                        estado_txt = "Acelerando";
                        color_estado = "#dc3545";
                    }
                } else {
                    f_fric = f_k; 
                    a = (P_x - f_fric) / m;
                    
                    if (a > 0) {
                        estado_txt = "Acelerando";
                        color_estado = "#dc3545";
                    } else if (a < -0.01) {
                        estado_txt = "Desacelerando";
                        color_estado = "#ffc107";
                    } else {
                        estado_txt = "Vel Constante";
                        color_estado = "#198754";
                        a = 0; // Estabilizar
                    }
                }
                
                v += a * dt;
                
                // No permitir subir por la pendiente espontáneamente (a menos que v > 0, pero aquí solo baja)
                if (v < 0) { 
                    v = 0; 
                    a = 0; 
                } 
                
                s += v * dt * 25; // factor visual 
                
                // BUCLE VISUAL: Al pasar de 260, reaparece en 10 (sin perder la variable 'v')
                if (s > 260) { 
                    s = 10; 
                }
                
                // Actualizar Panel HTML
                lbl_N.innerText = N_val.toFixed(1) + " N";
                lbl_Px.innerText = P_x.toFixed(1) + " N";
                lbl_fric.innerText = f_fric.toFixed(1) + " N";
                lbl_a.innerText = a.toFixed(2) + " m/s²";
                lbl_v.innerText = v.toFixed(2) + " m/s";
                lbl_estado.innerText = estado_txt;
                lbl_estado.style.color = color_estado;
            }

            function drawSim() {
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                
                // --- MITAD IZQUIERDA: PLANO INCLINADO ---
                const anchorX = 30;
                const anchorY = 60;
                const planeLen = 300;
                
                ctx.save();
                ctx.translate(anchorX, anchorY);
                ctx.rotate(theta_rad);
                
                // Superficie Inclinada
                ctx.beginPath();
                ctx.strokeStyle = "#888";
                ctx.lineWidth = 4;
                ctx.moveTo(0, 0);
                ctx.lineTo(planeLen, 0);
                ctx.stroke();
                
                // Rayado del piso
                ctx.beginPath();
                ctx.strokeStyle = "#bbb";
                ctx.lineWidth = 1;
                for (let i = 10; i < planeLen; i += 10) {
                    ctx.moveTo(i, 0);
                    ctx.lineTo(i - 8, 10);
                }
                ctx.stroke();

                // Bloque
                const blockW = 50;
                const blockH = 40;
                ctx.fillStyle = "#ff8c00"; 
                ctx.strokeStyle = "#c06b00";
                ctx.lineWidth = 2;
                ctx.fillRect(s, -blockH, blockW, blockH);
                ctx.strokeRect(s, -blockH, blockW, blockH);
                ctx.fillStyle = "white";
                ctx.font = "italic 20px serif";
                ctx.fillText("m", s + blockW/2 - 10, -12);
                
                ctx.restore();
                
                // Divisor Central
                ctx.beginPath();
                ctx.strokeStyle = "#e0e0e0";
                ctx.setLineDash([5, 5]);
                ctx.moveTo(canvas.width / 2 + 20, 20);
                ctx.lineTo(canvas.width / 2 + 20, canvas.height - 20);
                ctx.stroke();
                ctx.setLineDash([]);
                
                // --- MITAD DERECHA: DCL ROTADO ---
                const dcl_x = canvas.width * 0.77;
                const dcl_y = canvas.height / 2;
                const sf = 1.1; 
                
                ctx.save();
                ctx.translate(dcl_x, dcl_y);
                ctx.rotate(theta_rad);
                
                ctx.beginPath();
                ctx.strokeStyle = "#ccc";
                ctx.setLineDash([4, 4]);
                ctx.moveTo(-130, 0); ctx.lineTo(130, 0);   
                ctx.moveTo(0, -120); ctx.lineTo(0, 120);   
                ctx.stroke();
                ctx.setLineDash([]);
                
                if (chk_N.checked) drawArrow(ctx, 0, 0, 0, -N_val * sf, "#212529", 3); 
                if (chk_fric.checked && f_fric > 0) drawArrow(ctx, 0, 0, -f_fric * sf, 0, "#dc3545", 3); 
                
                if (chk_PxPy.checked) {
                    drawArrow(ctx, 0, 0, P_x * sf, 0, "#fd7e14", 2, true); 
                    drawArrow(ctx, 0, 0, 0, N_val * sf, "#fd7e14", 2, true); 
                }
                
                ctx.restore();
                
                if (chk_P.checked) {
                    drawArrow(ctx, dcl_x, dcl_y, dcl_x, dcl_y + P * sf, "#6f42c1", 3);
                }
                
                ctx.font = "bold 14px sans-serif";
                if (chk_N.checked) { 
                    ctx.fillStyle = "#212529"; 
                    ctx.fillText("N", dcl_x + N_val*sf*Math.sin(theta_rad) + 5, dcl_y - N_val*sf*Math.cos(theta_rad) - 5); 
                }
                if (chk_P.checked) { ctx.fillStyle = "#6f42c1"; ctx.fillText("P", dcl_x + 8, dcl_y + P * sf - 5); }
                if (chk_fric.checked && f_fric > 0) { 
                    ctx.fillStyle = "#dc3545"; 
                    ctx.fillText("f", dcl_x - f_fric*sf*Math.cos(theta_rad) - 15, dcl_y - f_fric*sf*Math.sin(theta_rad) - 5); 
                }
            }

            function animate(timestamp) {
                let dt = (timestamp - lastTime) / 1000;
                if (dt > 0.1) dt = 0.1; 
                lastTime = timestamp;
                
                if (isPlaying) {
                    updatePhysics(dt);
                }
                
                drawSim();
                animationId = requestAnimationFrame(animate);
            }

            ang_slider.addEventListener('input', () => {
                if (!isPlaying) {
                    updatePhysics(0);
                    drawSim();
                }
            });

            btn_play.addEventListener('click', () => {
                isPlaying = !isPlaying;
                if (isPlaying) {
                    btn_play.innerText = "⏸ Pausar";
                    btn_play.style.backgroundColor = "#ffc107";
                    btn_play.style.color = "#000";
                    lastTime = performance.now();
                } else {
                    btn_play.innerText = "▶ Reproducir";
                    btn_play.style.backgroundColor = "#198754";
                    btn_play.style.color = "#fff";
                }
            });

            btn_reset.addEventListener('click', () => {
                v = 0;
                s = 40;
                ang_slider.value = 0; // Opcional: Reiniciar el ángulo a 0 también
                if (!isPlaying) {
                    updatePhysics(0);
                    drawSim();
                }
            });
            
            [chk_N, chk_fric, chk_P, chk_PxPy].forEach(chk => {
                chk.addEventListener('change', () => {
                    if (!isPlaying) drawSim();
                });
            });
            
            btn_play.innerText = "⏸ Pausar";
            btn_play.style.backgroundColor = "#ffc107";
            btn_play.style.color = "#000";
            requestAnimationFrame(animate);
        }
        
        initSimulation();
    })();
</script>
"""

display(HTML(simulacion_html.replace('UID', uid)))